# Построение рекомендательной системы с помощью векторного представления графа

**Курсовая работа**

**Студент:** Безверхний, группа ПМ23-2

**Датасет:** Jester Dataset - https://goldberg.berkeley.edu/jester-data

**Репозиторий с кодом:** *<ссылка будет указана после публикации>*

---

## Содержание

1. Постановка задачи и метрики качества
2. Загрузка и предварительный анализ данных (EDA)
3. Подготовка данных и построение двудольного графа
4. Разделение на train / val / test
5. Архитектуры моделей: Popularity, MF-BPR, LightGCN, NGCF
6. Обучение моделей (BPR-loss, регуляризация, оценка валидации)
7. Сравнение моделей по метрикам Precision@K, Recall@K, NDCG@K
8. Подбор гиперпараметров (Grid Search)
9. Анализ результатов и выводы

## 1. Постановка задачи и метрики качества

**Объект исследования:** двудольный граф взаимодействий пользователей и шуток из Jester.

**Предмет исследования:** методы обучения векторных представлений вершин графа (graph embedding) для задачи top-K рекомендаций.

**Цель:** построить рекомендательную систему на векторных представлениях графа. Она должна предсказывать, какие шутки понравятся пользователю.

### Формулировка задачи
Есть множество пользователей $U$ и множество шуток $I$. Наблюдаемые взаимодействия $\mathcal{O} = \{(u, i, r_{ui})\}$, где $r_{ui} \in [-10, +10]$. Оценки переводим в неявную обратную связь: $y_{ui} = 1$ если $r_{ui} > 0$. Нужно обучить функцию ранжирования $f: U \times I \to \mathbb{R}$ так, чтобы для пользователя $u$ топ-K шуток с максимальным $f(u, i)$ содержал как можно больше реально понравившихся.

### Метрики качества (K = 10)
* **Precision@K** - доля релевантных шуток в первых K рекомендациях.
* **Recall@K** - доля найденных среди всех релевантных у пользователя.
* **NDCG@K** - Normalized Discounted Cumulative Gain, учитывает позиции в выдаче.
* **HitRate@K** - доля пользователей, у которых хотя бы одна релевантная шутка попала в топ-K.

### Ориентиры для метрик
В типичных recsys-задачах с неявной обратной связью на плотных матрицах (MovieLens-100K, Yelp-2018, Jester) LightGCN и NGCF обычно дают NDCG@10 от 0.35 до 0.50 и Recall@10 от 0.5 до 0.7 (см. статьи LightGCN-2020, NGCF-2019). Популярность на Jester должна обходить случайное угадывание в 5-10 раз. Графовая модель должна обогнать популярность по NDCG@10 на 5-15%.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import sys
sys.path.append('scripts')
%matplotlib inline
ART = Path('artifacts')


## 2. Предварительный анализ и очистка данных

Jester собран в Калифорнийском университете в Беркли. Содержит непрерывные оценки 100 шуток на шкале от -10 до +10. Распространяется тремя файлами: jester-data-1.xls, jester-data-2.xls, jester-data-3.xls. Пропускам соответствует код 99.

In [ ]:
from scripts.data_utils import load_and_split
df = pd.read_parquet('data/processed/ratings.parquet')
print('Всего строк:', len(df))
df.head()


### Базовая статистика

| Показатель | Значение |
|---|---|
| Пользователей | 73,421 |
| Шуток | 100 |
| Взаимодействий | 4,136,360 |
| Плотность матрицы | 0.563 (56.3%) |
| Среднее оценок на пользователя | 56.3 (медиана 52) |
| Среднее оценок на шутку | 41364 |
| Диапазон оценок | [-9.95, 10.00] |
| Среднее значение оценки | 0.742 |
| Стандартное отклонение | 5.295 |
| Доля положительных (rating > 0) | 0.585 |
| Доля высоких (rating > 5) | 0.252 |

**Особенность Jester.** Плотность около 56% - очень много по меркам recsys. Для сравнения, MovieLens-100K имеет около 6%, Yelp-2018 - меньше 0.1%. Это даёт три следствия: 
* Популярностный baseline и так очень сильный.
* Холодного старта по шуткам нет, каждая оценена десятками тысяч раз.
* Графовые методы получают много рёбер, но и преимущество над матричной факторизацией меньше.

### Графики разведочного анализа

![Распределение оценок. Бимодальное, с пиками возле -10 и +10.](artifacts/eda/rating_distribution.png)

*Распределение оценок. Бимодальное, с пиками возле -10 и +10.*

![Распределение числа оценок на пользователя.](artifacts/eda/user_activity.png)

*Распределение числа оценок на пользователя.*

![Популярность шуток (число оценок).](artifacts/eda/item_popularity.png)

*Популярность шуток (число оценок).*

![Средние рейтинги шуток.](artifacts/eda/item_mean_rating.png)

*Средние рейтинги шуток.*

![Фрагмент матрицы взаимодействий пользователь x шутка.](artifacts/eda/interaction_matrix.png)

*Фрагмент матрицы взаимодействий пользователь x шутка.*

## 3. Преобразование данных и построение графа

Матрица оценок превращается в **двудольный граф** $G = (U \cup I, E)$. Тут $U$ - узлы-пользователи, $I$ - узлы-шутки. Ребро $(u, i) \in E$ существует, если пользователь поставил шутке оценку $> 0$ (положительная имплицитная обратная связь).

Для графовых моделей (LightGCN, NGCF) нужна симметрично нормированная матрица смежности $\hat{A} = D^{-1/2} A D^{-1/2}$. Её размер - $(|U|+|I|) \times (|U|+|I|) = 73,521^2$. Лежит в памяти в разреженном COO-формате.

In [ ]:
from scripts.models import build_norm_adj
splits = load_and_split()
adj = build_norm_adj(splits.n_users, splits.n_items, splits.train_u, splits.train_i)
print('Размерность графа:', adj.shape)
print('Число рёбер (после симметризации):', adj._nnz())


## 4. Разделение выборки

Использован **стратифицированный по пользователю случайный split**. У каждого пользователя 10% его взаимодействий идёт в test, 10% в val, остальные 80% в train. Что это даёт:

* пользователь оказывается во всех трёх выборках, нет холодного старта;
* нет утечки данных между выборками;
* метрики статистически устойчивы.

Временной split тут невозможен - в Jester нет временных меток. Leave-one-out даёт слишком короткий вектор истины (одна шутка), это плохо сочетается с метриками типа Recall@10.

In [ ]:
print('users:', splits.n_users)
print('items:', splits.n_items)
print('train:', len(splits.train_u))
print('val:  ', len(splits.val_u))
print('test: ', len(splits.test_u))


## 5. Архитектуры моделей

Все четыре подхода лежат в одном модуле `scripts/models.py`. Фреймворк - PyTorch.

### 5.1 Popularity (baseline)
Без обучения. Скор шутки $i$ равен числу её положительных оценок в train. Все пользователи получают одинаковый порядок шуток.

### 5.2 MF-BPR (матричная факторизация с BPR-loss)
$$\hat{y}_{ui} = \mathbf{e}_u^T \mathbf{e}_i$$
Эмбеддинги пользователей и шуток обучаются через минимизацию:
$$\mathcal{L}_{BPR} = -\sum_{(u, i^+, i^-)} \log\sigma(\hat{y}_{ui^+} - \hat{y}_{ui^-}) + \lambda \| \theta \|^2$$

### 5.3 LightGCN
Графовая модель без обучаемых линейных преобразований. На слое $k$:
$$\mathbf{e}_v^{(k+1)} = \sum_{u \in N(v)} \frac{1}{\sqrt{|N(v)||N(u)|}} \mathbf{e}_u^{(k)}$$
Итоговый эмбеддинг - среднее по слоям $0..K$.

### 5.4 NGCF (Neural Graph Collaborative Filtering)
$$\mathbf{e}_v^{(k+1)} = \mathrm{LeakyReLU}\big( W_1^{(k)} \hat{A} E^{(k)} + W_2^{(k)} (\hat{A} E^{(k)} \odot E^{(k)}) \big)$$
Со $L_2$-нормализацией и dropout. Итог - конкатенация по слоям.

**Функция ошибки.** Для всех обучаемых моделей - BPR с $L_2$-регуляризацией ($\lambda = 10^{-5}$).
**Регуляризация.** $L_2$ на эмбеддинги, в NGCF дополнительно dropout = 0.1 и $L_2$-нормализация.


In [ ]:
from scripts.models import PopularityModel, MFBPR, LightGCN, NGCF
import inspect
for cls in [PopularityModel, MFBPR, LightGCN, NGCF]:
    print(cls.__name__, '- параметры:', list(inspect.signature(cls.__init__).parameters))


## 6. Обучение моделей

Параметры обучения:
* Оптимизатор: Adam (`lr = 5e-3`).
* Размер батча: 65 536. Большой батч сокращает число пропагаций по графу.
* Эпохи: 25 с ранней остановкой по NDCG@10 на валидации.
* Negative sampling: для каждого позитива (u, i) сэмплируется случайная шутка j ∉ train(u).
* Размерность эмбеддинга: 64.
* Число слоёв GCN: 3.

Воспроизводимый запуск:
```bash
python scripts/train.py --epochs 25 --dim 64 --layers 3 --batch 65536 --lr 0.005
```


In [ ]:
with open(ART/'all_models.json') as f:
    results = json.load(f)
print('Модели:', list(results.keys()))


## 7. Результаты обучения и сравнение моделей

| Модель | Precision@10 | Recall@10 | NDCG@10 | HitRate@10 | Время, с |
|---|---|---|---|---|---|
| **Popularity** | 0.1399 | 0.4850 | 0.3445 | 0.7515 | 0.0 |
| **MF-BPR** | 0.1532 | 0.5344 | 0.3623 | 0.7899 | 135.2 |
| **LightGCN** | 0.1617 | 0.5635 | 0.3860 | 0.8140 | 1471.0 |
| **NGCF** | 0.1549 | 0.5308 | 0.3616 | 0.7860 | 1850.1 |


**Лучшая модель по NDCG@10:** **LightGCN**.

Все обучаемые модели приходят примерно к одинаковому порядку метрик. Плотность Jester (56%) делает задачу не такой сложной: даже популярность даёт Recall@10 около 0.48. Графовые методы выигрывают на NDCG@10 за счёт персонализации: в popularity нет различий между пользователями, а тут они есть.

In [ ]:
# Кривые обучения
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, r in results.items():
    if not r['history']: continue
    h = pd.DataFrame(r['history'])
    axes[0].plot(h['epoch'], h['loss'], marker='o', label=name)
    axes[1].plot(h['epoch'], h['val_NDCG@10'], marker='o', label=name)
axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('BPR loss (train)'); axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].set_xlabel('Эпоха'); axes[1].set_ylabel('NDCG@10 (val)'); axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout(); plt.savefig('artifacts/learning_curves.png', dpi=130); plt.show()


In [ ]:
# Сравнительный график по метрикам
metrics = ['Precision@10', 'Recall@10', 'NDCG@10', 'HitRate@10']
names = list(results.keys())
data = np.array([[results[n]['test'][m] for m in metrics] for n in names])
x = np.arange(len(metrics)); w = 0.2
fig, ax = plt.subplots(figsize=(10, 5))
for j, n in enumerate(names):
    ax.bar(x + j*w - 1.5*w, data[j], w, label=n)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_title('Сравнение моделей по тестовым метрикам'); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig('artifacts/model_comparison.png', dpi=130); plt.show()


## 8. Подбор гиперпараметров (Grid Search)

Для лучшей модели (LightGCN) запущен автоматический поиск по сетке:

* `dim` ∈ {32, 64}
* `n_layers` ∈ {2, 3}
* `lr` ∈ {1e-3, 5e-3}
* `reg_w` ∈ {1e-5, 1e-4}

Всего 12 комбинаций по 8 эпох каждая. Запуск: `python scripts/grid_search.py`.

### Топ-10 конфигураций по NDCG@10 на val

| dim | n_layers | lr | reg_w | val NDCG@10 | test NDCG@10 |
|---|---|---|---|---|---|
| 64 | 2 | 5e-03 | 1e-04 | 0.3793 | 0.3774 |
| 64 | 2 | 5e-03 | 1e-05 | 0.3792 | 0.3772 |
| 32 | 2 | 5e-03 | 1e-05 | 0.3778 | 0.3755 |
| 32 | 2 | 5e-03 | 1e-04 | 0.3777 | 0.3754 |
| 32 | 3 | 5e-03 | 1e-05 | 0.3724 | 0.3711 |
| 32 | 3 | 5e-03 | 1e-04 | 0.3721 | 0.3709 |
| 64 | 2 | 1e-03 | 1e-05 | 0.3483 | 0.3482 |
| 64 | 2 | 1e-03 | 1e-04 | 0.3481 | 0.3480 |
| 32 | 2 | 1e-03 | 1e-05 | 0.3418 | 0.3418 |
| 32 | 2 | 1e-03 | 1e-04 | 0.3418 | 0.3417 |


**Лучшая конфигурация:** {'dim': 64, 'n_layers': 2, 'lr': 0.005, 'reg_w': 0.0001}, test NDCG@10 = **0.3774**.

## 9. Выводы

1. **Особенность датасета.** Jester очень плотный (около 56%) и с малым числом объектов (100 шуток). Это сильно отличает его от типичных recsys-бенчмарков: популярность тут сама по себе сильный baseline, потенциал персонализации ограничен.

2. **Сравнение моделей.** Из четырёх подходов лучший по NDCG@10 - **LightGCN**. Обучаемые модели стабильно бьют popularity-baseline. А вот между MF-BPR, LightGCN и NGCF разрыв небольшой. Граф из 100 шуток слабо использует преимущества многослойной агрегации.

3. **Проверка переобучения.** NDCG@10 на валидации показывает, что best epoch у каждой модели достигается раньше конца обучения (см. поле `best_epoch`). В тест идёт чекпоинт с лучшей валидацией, переоценки нет.

4. **Подбор гиперпараметров.** Grid Search по четырём параметрам (dim, n_layers, lr, reg_w) показал, что сильнее всего на качество влияют learning rate и число слоёв. L2-регуляризация почти не двигает результат, потому что плотные данные сами по себе ограничивают переобучение.

5. **Перспективы улучшения.**
    * Не бинаризовать оценки, а использовать взвешенные рёбра (rating-aware loss).
    * Добавить контрастивное предобучение (SGL, SimGCL) - особенно полезно при шумных оценках.
    * Подмешать признаки шуток (длина текста, темы) в духе PinSAGE.
    * На датасетах с большим $|I|$ имеет смысл использовать GraphSAGE-семплирование соседей.

### Список использованных источников

1. Jester Dataset homepage - https://goldberg.berkeley.edu/jester-data/
2. PyTorch documentation - https://pytorch.org/docs/stable/
3. NumPy documentation - https://numpy.org/doc/stable/
4. Pandas documentation - https://pandas.pydata.org/docs/
5. SciPy Sparse documentation - https://docs.scipy.org/doc/scipy/reference/sparse.html
6. Matplotlib documentation - https://matplotlib.org/stable/index.html
7. He X. et al. LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation. arXiv: https://arxiv.org/abs/2002.02126
8. Wang X. et al. Neural Graph Collaborative Filtering. arXiv: https://arxiv.org/abs/1905.08108
9. Rendle S. et al. BPR: Bayesian Personalized Ranking from Implicit Feedback. arXiv: https://arxiv.org/abs/1205.2618
10. Towards Data Science. A gentle introduction to LightGCN - https://towardsdatascience.com/lightgcn-with-pytorch-geometric-91bab836471e
